In [ ]:
## 1. Project Initialization
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Apply a clean, modern aesthetic style for all subsequent visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.weight'] = 'bold'

print("Libraries imported successfully and global plotting canvas styles established!")

In [ ]:
Data Loading & Fail-Safe Ingestion

In [ ]:
## 2. Load the Datasets
# Using try-except blocks ensures that if a file is missing, the notebook flags it clearly without crashing structural memory.
try:
    points = pd.read_csv('points_table.csv')
    print("✓ Points table loaded successfully.")
except FileNotFoundError:
    print("⚠ 'points_table.csv' not found. Please verify the filename in your workspace.")

try:
    batting_stats = pd.read_csv('batting_stats.csv')
    print("✓ Batting statistics loaded successfully.")
except FileNotFoundError:
    print("⚠ 'batting_stats.csv' not found.")

In [ ]:
The Points Table Standings Visualization

In [ ]:
## 3. Points Table Standings Analysis

# Sort teams hierarchically by total points first, then resolve ties using Net Run Rate (NRR)
points_sorted = points.sort_values(by=['Points', 'NRR'], ascending=[False, False]).reset_index(drop=True)

print("--- IPL 2026 League Standings ---")
print(points_sorted[['Pos', 'Team', 'Pld', 'Won', 'Lost', 'Points', 'NRR']])

# Plotting the standings
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    x='Points', 
    y='Team', 
    data=points_sorted, 
    palette='viridis',
    edgecolor='black'
)

# Dynamically annotate the right side of each bar with its exact points value and signed NRR
for i, p in enumerate(ax.patches):
    width = p.get_width()
    nrr_val = points_sorted.loc[i, 'NRR']
    label_text = f" {int(width)} pts (NRR: {nrr_val:+.3f})"
    
    ax.text(
        width + 0.2, 
        p.get_y() + p.get_height() / 2, 
        label_text, 
        va='center', 
        fontsize=11, 
        fontweight='bold'
    )

plt.title('IPL 2026 Points Table & Team Standings', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Points Secured', fontsize=12, fontweight='bold')
plt.ylabel('Franchise Team', fontsize=12, fontweight='bold')
plt.xlim(0, points_sorted['Points'].max() + 4)

sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
Batting Impact Profiles (Volume vs. Tempo)

In [7]:
## 4. Top Run Scorers and Strike Rate Overlay

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- GUARANTEED DATA LOADING ---
# We will look for common filenames and load it directly here so it never throws a NameError
possible_files = ['batting_stats.csv', 'batting.csv', 'IPL_2026_batting.csv']
batting_df = None

for file in possible_files:
    if os.path.exists(file):
        batting_df = pd.read_csv(file)
        print(f"✓ Successfully loaded data from local file: '{file}'")
        break

if batting_df is None:
    raise FileNotFoundError("Could not find your batting CSV file. Please ensure it's named 'batting_stats.csv' or 'batting.csv' in this folder!")

# --- DATA PROCESSING ---
# Filter out the top 10 batters by total scoring volume
top_batters = batting_df.sort_values(by='Runs', ascending=False).head(10).reset_index(drop=True)

print("\n--- IPL 2026 Orange Cap Frontrunners ---")
print(top_batters[['Player', 'Team', 'Matches', 'Runs', 'StrikeRate']])

# --- VISUALIZATION CODE ---
fig, ax1 = plt.subplots(figsize=(14, 7))

# Primary Y-Axis: Bar plot representing total accumulation of runs
sns.barplot(
    x='Player', 
    y='Runs', 
    data=top_batters, 
    palette='magma', 
    alpha=0.85, 
    ax=ax1,
    edgecolor='black'
)
ax1.set_title('IPL 2026 Elite Batting Profiles: Volume vs. Tempo', fontsize=16, fontweight='bold', pad=25)
ax1.set_xlabel('Batsman', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Runs Scored (Bars)', fontsize=12, fontweight='bold', color='purple')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=30, ha='right', fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='purple')

# Display exact run integers within the interior bounds of the bars
for p in ax1.patches:
    ax1.annotate(
        f"{int(p.get_height())}", 
        (p.get_x() + p.get_width() / 2., p.get_height() - 40), 
        ha='center', va='center', 
        color='white', fontsize=11, fontweight='bold'
    )

# Secondary Y-Axis: Line graph charting striking efficiency across the same group
ax2 = ax1.twinx()
sns.lineplot(
    x='Player', 
    y='StrikeRate', 
    data=top_batters, 
    color='crimson', 
    marker='o', 
    linewidth=3, 
    markersize=8, 
    ax=ax2,
    label='Strike Rate'
)
ax2.set_ylabel('Batting Strike Rate (Line)', fontsize=12, fontweight='bold', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.grid(False)

# Highlight Vaibhav Sooryavanshi if he is in the top 10
if 'Vaibhav Sooryavanshi' in top_batters['Player'].values:
    v_idx = top_batters[top_batters['Player'] == 'Vaibhav Sooryavanshi'].index[0]
    v_sr = top_batters.loc[v_idx, 'StrikeRate']
    
    ax2.annotate(
        f"Historic Season!\nSR: {v_sr:.1f}",
        xy=(v_idx, v_sr),
        xytext=(v_idx + 0.4, v_sr + 15),
        arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.5),
        fontsize=10, color='darkred', fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.7, ec="orange")
    )

plt.tight_layout()
plt.show()

FileNotFoundError: Could not find your batting CSV file. Please ensure it's named 'batting_stats.csv' or 'batting.csv' in this folder!

In [ ]:
Advanced Boundaries & Boundary Contribution Rate

In [6]:
## 5. Aggressive Intent Matrix: Sixes vs. Boundary Percentage

import pandas as pd
import matplotlib.pyplot as plt

try:
    batting
except NameError:
    batting = pd.read_csv('batting_stats.csv')

# Calculate what percentage of a player's runs come purely from boundaries
if 'Fours' in batting.columns and 'Sixes' in batting.columns:
    batting['BoundaryRuns'] = (batting['Fours'] * 4) + (batting['Sixes'] * 6)
    batting['BoundaryPct'] = (batting['BoundaryRuns'] / batting['Runs']) * 100

top_six_hitters = batting.sort_values(by='Sixes', ascending=False).head(10)

plt.figure(figsize=(12, 6))
scatter = plt.scatter(
    batting['StrikeRate'], 
    batting['Sixes'], 
    s=batting['Runs']*0.5, # Size relates to their total run volume
    c=batting['BoundaryPct'] if 'BoundaryPct' in batting.columns else None,
    cmap='coolwarm', 
    alpha=0.75, 
    edgecolors='black'
)

# Label top historical performers directly on the scatter plane
for idx, row in top_six_hitters.head(5).iterrows():
    plt.text(
        row['StrikeRate'] + 2, 
        row['Sixes'], 
        row['Player'], 
        fontsize=10, 
        fontweight='bold'
    )

plt.title('IPL 2026 Intent Evaluation: Maxima Distribution Map', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Batting Strike Rate', fontsize=12, fontweight='bold')
plt.ylabel('Total Sixes Smashed', fontsize=12, fontweight='bold')

if 'BoundaryPct' in batting.columns:
    cbar = plt.colorbar(scatter)
    cbar.set_label('% of Total Runs Scored in Boundaries', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'batting_stats.csv'